# Revenue vs Spending, and Growth vs Population + Inflation
How does state revenue growth compare to what inflation and population growth alone would explain?

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
rev = pd.read_csv(PROCESSED / "state_revenue.csv")
exp = pd.read_csv(PROCESSED / "state_expenditures.csv")
mac = pd.read_csv(PROCESSED / "macro.csv").set_index("fy")
T = rev[(rev.group != "Transfers")].groupby("fy").amount_k.sum()
E = exp[exp.fund.isin(["General Fund", "Special Funds"])].groupby("fy").amount_k.sum()
df = pd.DataFrame({"revenue": T, "spending": E}).dropna()
df = df.join(mac[["cpi_factor", "population_k"]])
df["rev_real_b"] = df.revenue * df.cpi_factor / 1e6
df["spend_real_b"] = df.spending * df.cpi_factor / 1e6
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df.rev_real_b, name="Revenue", line=dict(color=PALETTE[0], width=2.5)))
fig.add_trace(go.Scatter(x=df.index, y=df.spend_real_b, name="Spending", line=dict(color=PALETTE[2], width=2.5)))
fig.update_layout(yaxis_title=f"{base_fy()} $B", xaxis_title="Fiscal year (ending June 30)", yaxis_tickformat="$,.0f")
fig.add_vrect(x0=LAST_ACTUAL_FY + 0.5, x1=df.index.max(), fillcolor="grey", opacity=0.12, line_width=0, annotation_text="estimate")
base_layout(fig, "State revenue vs state-funded spending (real $B)", "General + special funds. Spending excludes bond and federal funds, so the gap is not a deficit. " + SOURCE_NOTE)
fig.show()

In [2]:
START = 1985
idx = pd.DataFrame({
    "Nominal revenue": df.revenue,
    "Real revenue per person": df.revenue * df.cpi_factor / df.population_k,
    "Real spending per person": df.spending * df.cpi_factor / df.population_k,
    "Population": df.population_k,
    "Prices (CPI-U)": 1 / df.cpi_factor,
}).loc[START:]
idx = idx / idx.iloc[0] * 100
fig = go.Figure()
for i, c in enumerate(idx.columns):
    fig.add_trace(go.Scatter(x=idx.index, y=idx[c], name=c, mode="lines", line=dict(color=PALETTE[i], width=2.5)))
fig.update_layout(yaxis_title=f"Index ({START} = 100)", xaxis_title="Fiscal year (ending June 30)", yaxis_type="log")
fig.add_vrect(x0=LAST_ACTUAL_FY + 0.5, x1=idx.index.max(), fillcolor="grey", opacity=0.12, line_width=0)
base_layout(fig, f"Growth since {START}: revenue, spending, population and prices", "Log scale. " + SOURCE_NOTE)
fig.show()